# Prepping Data Challenge - Week 

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
undergrad_loans = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_12_Undergraduate_Loans.csv')
repayments = pd.read_csv('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_12_Repayment.csv')

In [3]:
undergrad_loans.head()

,Loan Type,Course Start,Course Length (years),Amount per year
0,Tuition Fees,September 2020,3,9250
1,Maintenance,September 2020,3,5820


In [4]:
repayments.head()

,Loan,Interest,Repayment Threshold,% Repayment over Threshold
0,Undergraduate,7.70%,27295,9
1,Postgraduate,7.70%,21000,6


In [5]:
print('\nDataFrame Shapes:')
print(f'  Undergraduate Loans:{undergrad_loans.shape}')
print(f'  Repayments:{repayments.shape}')


DataFrame Shapes:
  Undergraduate Loans:(2, 4)
  Repayments:(2, 4)


## Challenges

### Produce a set of payout dates and calculate the number of months until April 2024 

In [6]:
APRIL_24 = np.datetime64('2024-04-01')

In [7]:
REPAYMENT_DATE = '09-01'
payout_dates = pd.DataFrame({'Payment Date' : [np.datetime64(f'{year}-{REPAYMENT_DATE}') for year in [2020, 2021, 2022]]})

In [8]:
payout_dates

,Payment Date
0,2020-09-01
1,2021-09-01
2,2022-09-01


In [9]:
payout_dates['Months to current Date'] = (APRIL_24 - payout_dates['Payment Date']) // pd.Timedelta(days=30.25)

### Add the Fees for the course

In [10]:
payout_dates['Fees'] = undergrad_loans['Amount per year'].sum()

In [11]:
payout_dates

,Payment Date,Months to current Date,Fees
0,2020-09-01,43,15070
1,2021-09-01,31,15070
2,2022-09-01,19,15070


### Calculate the interest accumulated for each payment (applied monthly)

In [12]:
INTEREST_RATE = repayments['Interest'].apply(lambda x: float(x[:-1]) / 100)[0]

In [13]:
payout_dates['Interest of Payment'] = payout_dates['Fees'] * (1 + INTEREST_RATE / 12)**payout_dates['Months to current Date']

### Total the values into a new DataFrame

In [14]:
repayment_data = pd.DataFrame({'Total Borrowed' : payout_dates['Fees'].sum(),
 'Total Borrowed + Interest' : payout_dates['Interest of Payment'].sum()}, index=[0])

### Add a Salary parameter, figure out the monthly payment amount and the amount of interest added after repayment

#### Salary and graduation level parameter

In [15]:
SALARY = 35000
GRAD_TYPE = 'Undergraduate'

### Calculate repayment amount

Divide by 1200 to get percent and payment per month

In [16]:
repayment_data['Monthly Repayment'] = (SALARY - repayments[repayments['Loan'] == GRAD_TYPE]['Repayment Threshold']) * repayments[repayments['Loan'] == GRAD_TYPE]['% Repayment over Threshold'] / 1200

### Monthly Interest added after payment

In [17]:
repayment_data['Interest added next Month'] = (repayment_data['Total Borrowed + Interest'] - repayment_data['Monthly Repayment']) * INTEREST_RATE / 12

In [18]:
repayment_data

,Total Borrowed,Total Borrowed + Interest,Monthly Repayment,Interest added next Month
0,45210,55233.091467,57.7875,354.041534


## Save Results

In [19]:
repayment_data.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/12_loan_payments.csv')